In [0]:
#Notebook intended to run within the databricks INT environment; should work in REACH too but untested

In [0]:
%pip install openpyxl

  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/250.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 204.8/250.9 kB 6.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 5.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
import pandas as pd
import re


In [0]:
#Set your database and output path
database = "data_mgmt.derived"
output_path = "pmap_data_dictionary.xlsx"  # Local driver path
#output_path = "test.xlsx"
#Get the list of tables
tables_df = spark.sql(f"SHOW TABLES IN {database}")
tables = [row["tableName"] for row in tables_df.collect()]
#tables

In [0]:
#Helper to generate a valid, unique Excel sheet name (<= 31 chars, no invalid characters)
def make_sheet_name(name, existing):
  # Remove invalid characters: : \ / ? * [ ]
  clean = re.sub(r'[:\/?*$$]', '', name)
  # Excel sheet name length limit
  base = clean[:31]
  sheet = base
  i = 1
  # Ensure uniqueness
  while sheet in existing:
    suffix = f"{i}"
    sheet = (base[: 31 - len(suffix)]) + suffix
    i += 1
    existing.add(sheet)
  return sheet


In [0]:
#tables = tables[0:2] #For testing
#Create an Excel writer and write each DESCRIBE EXTENDED result to its own sheet
existing_sheet_names = set()
with pd.ExcelWriter(output_path, mode='w') as writer:
  for tbl in tables:
    full_table_name = f"{database}.{tbl}"
    try:
      df = spark.sql(f"DESCRIBE {full_table_name}")
      #display(df)
      # Convert to pandas for Excel
      pdf = df.toPandas()
      sheet_name = make_sheet_name(tbl, existing_sheet_names)
      pdf.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
      # Optionally record errors in a separate sheet
      err_pdf = pd.DataFrame({"table": [full_table_name], "error": [str(e)]})
      sheet_name = make_sheet_name(f"{tbl}_ERROR", existing_sheet_names)
      err_pdf.to_excel(writer, sheet_name=sheet_name, index=False)
print(f"Excel file written to: {output_path}")
#print("If you want to download from Databricks, you can use:")
#print(f"displayHTML('<a href="files{output_path[5:]}">Download Excel</a>')")  # converts /dbfs/... to /files/... for browser download in notebooks

Excel file written to: pmap_data_dictionary.xlsx
